In [2]:
import numpy as np
import pandas as pd
import re

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Preparacion de datos

In [5]:
# df = pd.read_csv('data/cleaned.csv')
# X_train, X_test = train_test_split(df, test_size=0.2, stratify=df['Modelo'], random_state=42)
# # Save X_test to a CSV file inside the data folder
# X_test.to_csv('data/cleaned_test.csv', index=True)
# print(f"X_test dataset with {X_test.shape[0]} rows and {X_test.shape[1]} columns saved to data/cleaned_test.csv")
# X_train.to_csv('data/cleaned_train.csv', index=True)
# print(f"X_train dataset with {X_train.shape[0]} rows and {X_train.shape[1]} columns saved to data/cleaned_train.csv")

In [7]:
df = pd.read_csv('data/train/cleaned_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Año                      14603 non-null  float64
 4   Versión                  14085 non-null  object 
 5   Color                    14288 non-null  object 
 6   Tipo de combustible      14603 non-null  object 
 7   Puertas                  14603 non-null  float64
 8   Transmisión              14590 non-null  object 
 9   Motor                    14410 non-null  float64
 10  Tipo de carrocería       14603 non-null  object 
 11  Con cámara de retroceso  3749 non-null   object 
 12  Kilómetros               14603 non-null  float64
 13  Título                   14603 non-null  object 
 14  Precio                

In [ ]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

KeyError: "['Unnamed: 0'] not found in axis"

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11682 entries, 12283 to 7270
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Marca                        11682 non-null  float64
 1   Modelo                       11682 non-null  float64
 2   Año                          11682 non-null  float64
 3   Versión                      11682 non-null  float64
 4   Color                        11682 non-null  float64
 5   Puertas                      11682 non-null  float64
 6   Motor                        11682 non-null  float64
 7   Kilómetros                   11682 non-null  float64
 8   Moneda                       11682 non-null  float64
 9   Turbo                        11682 non-null  float64
 10  cv                           11682 non-null  float64
 11  Tipo de combustible_0.0      11682 non-null  float64
 12  Tipo de combustible_1.0      11682 non-null  float64
 13  Tipo de combustibl

In [ ]:
X_train.describe()

,Marca,Modelo,Año,Versión,Color,Puertas,Motor,Kilómetros,Moneda,Turbo,...,Con cámara de retroceso_1.0,Tracción_0.0,Tracción_0.2,Tracción_0.4,Tracción_0.6,Tracción_0.8,Tracción_1.0,Tipo de vendedor_0.0,Tipo de vendedor_1.0,Tipo de vendedor_2.0
count,1.168200e+04,1.168200e+04,1.168200e+04,11682.000000,11682.000000,1.168200e+04,11682.000000,1.168200e+04,1.168200e+04,1.168200e+04,...,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000
mean,4.196837e-17,8.226409e-17,3.223657e-17,-0.009432,0.000111,3.041186e-18,0.003401,2.554596e-17,-9.610148e-17,-2.432949e-18,...,0.521743,0.491269,0.082435,0.070536,0.071306,0.069081,0.215374,0.465588,0.308423,0.225989
std,1.000043e+00,1.000043e+00,1.000043e+00,0.985434,0.991894,1.000043e+00,0.998256,1.000043e+00,1.000043e+00,1.000043e+00,...,0.499548,0.499945,0.275037,0.256059,0.257347,0.253602,0.411099,0.498836,0.461862,0.418250
min,-1.885412e+00,-1.447198e+00,-1.869640e-02,-1.284952,-6.739097,-1.909881e-02,-1.306811,-9.630825e-01,-7.875593e-01,-7.265382e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.275336e-01,-5.903493e-01,-9.993179e-03,-0.558234,0.097418,-1.599110e-02,-0.503151,-9.625953e-01,-7.875593e-01,-7.265382e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-4.139938e-01,-2.889810e-01,-9.247188e-03,-0.259900,0.104770,-1.599110e-02,-0.342419,-8.552711e-02,-7.875593e-01,-7.265382e-01,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.382134e-01,4.650262e-01,-8.003871e-03,0.255500,0.701287,-1.599110e-02,0.300509,5.210185e-01,1.269746e+00,1.376390e+00,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,6.153100e+00,8.340739e+00,1.080786e+02,15.225353,14.233455,6.239391e+01,7.372717,2.097580e+01,1.269746e+00,1.376390e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Base Line

In [ ]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Base Line")
mlflow.set_tracking_uri("http://localhost:5000")

2025/06/13 12:04:16 INFO mlflow.tracking.fluent: Experiment with name 'Base Line' does not exist. Creating a new experiment.


In [ ]:
reports = []

## Regresor lineal

In [ ]:
# Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [ ]:
# Predict on the validation set
y_pred_train_lr = lr.predict(X_train)
y_pred_val_lr = lr.predict(X_val)

mse_train_lr = mean_squared_error(y_train, y_pred_train_lr)
r2_train_lr = r2_score(y_train, y_pred_train_lr)

mse_val_lr = mean_squared_error(y_val, y_pred_val_lr)
r2_val_lr = r2_score(y_val, y_pred_val_lr)

print(f"MSE (train): {mse_train_lr:.4f}")
print(f"R² (train): {r2_train_lr:.4f}")
print(f"MSE (val): {mse_val_lr:.4f}")
print(f"R² (val): {r2_val_lr:.4f}")

reports.append({
    "model": "Linear Regression",
    "params": None,
    "mse_train": mse_train_lr,
    "r2_train": r2_train_lr,
    "mse_val": mse_val_lr,
    "r2_val": r2_val_lr
})

MSE (train): 76327988305835.2656
R² (train): 0.8478
MSE (val): 261905055618799.6250
R² (val): 0.5982


## Random Forest

In [ ]:
# Definir los hiperparámetros en un diccionario
rf_params = {
    'n_estimators': 100,       # cantidad de árboles
    'max_depth': None,         # sin profundidad máxima (puede sobreajustar si no se regula)
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42,
    'n_jobs': -1               # usar todos los núcleos disponibles
}

In [ ]:
# Crear el modelo Random Forest
rf_reg = RandomForestRegressor(**rf_params)
rf_reg.fit(X_train, y_train)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [ ]:

# Predecir en el set de validación
y_pred_train_rf = rf_reg.predict(X_train)
y_pred_val_rf = rf_reg.predict(X_val)

# Evaluar el modelo
mse_train_rf = mean_squared_error(y_train, y_pred_train_rf)
r2_train_rf = r2_score(y_train, y_pred_train_rf)

mse_val_rf = mean_squared_error(y_val, y_pred_val_rf)
r2_val_rf = r2_score(y_val, y_pred_val_rf)

print(f"MSE (train): {mse_train_rf:.4f}")
print(f"R² (train): {r2_train_rf:.4f}")
print(f"MSE (val): {mse_val_rf:.4f}")
print(f"R² (val): {r2_val_rf:.4f}")

MSE (train): 12258456536346.9512
R² (train): 0.9756
MSE (val): 150982332868335.3125
R² (val): 0.7684


In [ ]:
reports.append({
    "model": "Random Forest",
    "params": rf_params,
    "mse_train": mse_train_rf,
    "r2_train": r2_train_rf,
    "mse_val": mse_val_rf,
    "r2_val": r2_val_rf
})

## XGBoost

In [ ]:
# Definir los parámetros en un diccionario
xgb_params = {
    'objective': 'reg:squarederror',  # objetivo para regresión
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'random_state': 42
}

In [ ]:
# Crear el modelo regresor con los parámetros
xgb_reg = XGBRegressor(**xgb_params)
xgb_reg.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:

y_pred_train_xgb = xgb_reg.predict(X_train)
y_pred_val_xgb = xgb_reg.predict(X_val)

# Evaluar el modelo con métricas de regresión
mse_train_xgb = mean_squared_error(y_train, y_pred_train_xgb)
r2_train_xgb = r2_score(y_train, y_pred_train_xgb)

mse_val_xgb = mean_squared_error(y_val, y_pred_val_xgb)
r2_val_xgb = r2_score(y_val, y_pred_val_xgb)

print(f"MSE (train): {mse_train_xgb:.4f}")
print(f"R² (train): {r2_train_xgb:.4f}")
print(f"MSE (val): {mse_val_xgb:.4f}")
print(f"R² (val): {r2_val_xgb:.4f}")

MSE (train): 27576898763307.0625
R² (train): 0.9450
MSE (val): 141598039228799.0312
R² (val): 0.7828


In [ ]:

reports.append({
    "model": "XGBoost",
    "params": xgb_params,
    "mse_train": mse_train_xgb,
    "r2_train": r2_train_xgb,
    "mse_val": mse_val_xgb,
    "r2_val": r2_val_xgb
})

## Red neuronal

In [ ]:
# Definir los hiperparámetros en un diccionario
mlp_params = {
    'hidden_layer_sizes': (100, 50),   # dos capas ocultas con 100 y 50 neuronas
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,                   # L2 regularization
    'learning_rate': 'adaptive',
    'max_iter': 500,
    'random_state': 42
}

In [ ]:
# Crear el modelo MLP
mlp_reg = MLPRegressor(**mlp_params)

mlp_reg.fit(X_train, y_train)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor(hidden_layer_sizes=(100, 50), learning_rate='adaptive',
             max_iter=500, random_state=42)

In [ ]:
y_pred_train_mlp = mlp_reg.predict(X_train)
y_pred_val_mlp = mlp_reg.predict(X_val)

# Evaluar el modelo
mse_train_mlp = mean_squared_error(y_train, y_pred_train_mlp)
r2_train_mlp = r2_score(y_train, y_pred_train_mlp)

mse_val_mlp = mean_squared_error(y_val, y_pred_val_mlp)
r2_val_mlp = r2_score(y_val, y_pred_val_mlp)

print(f"MSE (train): {mse_train_mlp:.4f}")
print(f"R² (train): {r2_train_mlp:.4f}")
print(f"MSE (val): {mse_val_mlp:.4f}")
print(f"R² (val): {r2_val_mlp:.4f}")

MSE (train): 79060413426572.3750
R² (train): 0.8423
MSE (val): 211666291809924.5938
R² (val): 0.6753


In [ ]:
reports.append({
    "model": "MLP",
    "params": mlp_params,
    "mse_train": mse_train_mlp,
    "r2_train": r2_train_mlp,
    "mse_val": mse_val_mlp,
    "r2_val": r2_val_mlp
})

In [ ]:
input_example = X_train.iloc[0:1]

for report in reports:
    with mlflow.start_run(run_name=report["model"]):
        if report["params"]:
            mlflow.log_params(report["params"])
        mlflow.log_metrics({
            "mse_train": report["mse_train"],
            "r2_train": report["r2_train"],
            "mse_val": report["mse_val"],
            "r2_val": report["r2_val"]
        })
        mlflow.log_artifact("data/cleaned_train.csv")
        signature = infer_signature(input_example, report["model"].predict(input_example))
        if report["model"] == "XGBoost":
            mlflow.xgboost.log_model(xgb_reg, "model", input_example=input_example, signature=signature)
        else:
            mlflow.sklearn.log_model(report["model"], "model", input_example=input_example, signature=signature)

NameError: name 'X_train' is not defined